In [22]:
import os 
from dotenv import load_dotenv
load_dotenv()
from langchain_community.retrievers import PineconeHybridSearchRetriever
from pinecone import Pinecone , ServerlessSpec
index_name = "hybrid-search-langchain-pinecone"

# creating a pinecone object 
pc = Pinecone(api_key = os.getenv("PINECONE_API_KEY"))

# create the index 
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name = index_name,
        dimension = 384, # dimension of dense vector
        metric="dotproduct", # sparse values supported for dotproduct 
        spec=ServerlessSpec(cloud="aws" , region="us-east-1")
    )

In [23]:
index = pc.Index(index_name)
index

In [24]:
## vector embedding and sparse matrix 
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")
embeddings = HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2")
embeddings

HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [25]:
from pinecone_text.sparse import BM25Encoder # use tfidf by default

bm25_encoder = BM25Encoder().default()
bm25_encoder

In [26]:
sentences = [
    "in 2021 i visited united kingdom",
    "in 2022 i visited america",
    "in 2023 i visited India",
]
# tfidf values on this 
bm25_encoder.fit(sentences)

# store the values to a json file 
bm25_encoder.dump("bm25_values.json")

100%|██████████| 3/3 [00:00<00:00, 650.35it/s]

In [27]:
# new retriever that supports both search 
retriever = PineconeHybridSearchRetriever(embeddings=embeddings , sparse_encoder=bm25_encoder , index=index)
retriever

PineconeHybridSearchRetriever(embeddings=HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False), sparse_encoder=<pinecone_text.sparse.bm25_encoder.BM25Encoder object at 0x000002D9AF255F10>, index=<pinecone.data.index.Index object at 0x000002D9E1504FE0>)

In [28]:
# adding sentences // inserting 
retriever.add_texts(
    sentences
)

100%|██████████| 1/1 [00:02<00:00,  2.92s/it]


In [29]:
retriever.invoke(
    "which city did i visit in recent"
)

[Document(metadata={'score': 0.260178179}, page_content='in 2021 i visited united kingdom'),
 Document(metadata={'score': 0.255768657}, page_content='in 2021 i visited america'),
 Document(metadata={'score': 0.231446654}, page_content='in 2021 i visited India'),
 Document(metadata={'score': 0.194889128}, page_content='in 2022 i visited america')]